# Module D — Ranking, Scoring, & Evaluation

1. Translation Failures
2. Named Entity Mismatch
3. Semantic vs. Lexical Wins
4. Cross-Script Ambiguity
5. Code-Switching

In [1]:
import feedparser
import json
from tqdm import tqdm
import os
from bs4 import BeautifulSoup
import html
import re
import pickle
from rank_bm25 import BM25Okapi

## 4. BN <-> EN CLIR

### 4.1 Load Saved BM25 Indexes

In [2]:
def load_index(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

en_pack = load_index('bm25_en.pkl')
bn_pack = load_index('bm25_bn.pkl')

bm25_en = en_pack['bm25']
doc_ids_en = en_pack['doc_ids']
docs_en = en_pack['docs']

bm25_bn = bn_pack['bm25']
doc_ids_bn = bn_pack['doc_ids']
docs_bn = bn_pack['docs']


def tokenize_en(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def tokenize_bn(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def search_en(query, top_k=5):
    q = tokenize_en(query)
    scores = bm25_en.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_en[i], "score": float(scores[i]), "title": docs_en[i].get("title",""), "url": docs_en[i].get("url","")} for i in idx]

def search_bn(query, top_k=5):
    q = tokenize_bn(query)
    scores = bm25_bn.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_bn[i], "score": float(scores[i]), "title": docs_bn[i].get("title",""), "url": docs_bn[i].get("url","")} for i in idx]

print("Index loaded:" , len(docs_en), "English documents and", len(docs_bn), "Bengali documents.")


Index loaded: 153 English documents and 164 Bengali documents.


### 4.2 Language Detection (BN vs EN)

In [3]:
def is_bangla(text):
    for ch in text:
        o = ord(ch) #unicode of bangla
        if 0x0980 <= o <= 0x09FF:
            return True
    return False    

In [4]:
is_bangla("এই একটি বাংলা বাক্য।")
is_bangla("this is an english sentence.")

False

### 4.3 Load BN <-> EN translation model (MarianMT / OPUS MT)

In [5]:
from transformers import MarianMTModel, MarianTokenizer

BN_EN_NAME = "Helsinki-NLP/opus-mt-bn-en"
EN_BN_NAME = "shhossain/opus-mt-en-to-bn"

tok_bn_en = MarianTokenizer.from_pretrained(BN_EN_NAME)
mod_bn_en = MarianMTModel.from_pretrained(BN_EN_NAME)

tok_en_bn = MarianTokenizer.from_pretrained(EN_BN_NAME)
mod_en_bn = MarianMTModel.from_pretrained(EN_BN_NAME)

c:\Users\ASUS\miniconda3\envs\clir_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def translate_bn_to_en(text):
    batch = tok_bn_en([text], return_tensors="pt",padding=True, truncation=True)
    gen = mod_bn_en.generate(**batch, max_new_tokens=128)
    return tok_bn_en.batch_decode(gen, skip_special_tokens=True)[0]

print("bn_to_en Translation models loaded.")


bn_to_en Translation models loaded.


In [7]:
def translate_en_to_bn(text):
    batch = tok_en_bn([text], return_tensors="pt",padding=True, truncation=True)
    gen = mod_en_bn.generate(**batch, max_new_tokens=128)
    return tok_en_bn.batch_decode(gen, skip_special_tokens=True)[0]

print("en_to_bn Translation models loaded.")

en_to_bn Translation models loaded.


In [8]:
print(translate_bn_to_en("বাংলাদেশ একটি সুন্দর দেশ।"))
print(translate_en_to_bn("Bangladesh is a beautiful country."))

Bangladesh is a beautiful country.
বাংলাদেশ একটি সুন্দর দেশ।


### 4.4 CLIR Search Function

In [9]:
def clir_search(query, top_k=5):
    if is_bangla(query):
        q_en = translate_bn_to_en(query)
        results_bn = search_bn(query, top_k)
        results_en = search_en(q_en, top_k)
        return_en = {"queary_language": "bn", "translated_query": q_en, "results_language": "en", "results": results_en}
        return_bn = {"queary_language": "bn", "translated_query": q_en, "results_language": "bn", "results": results_bn}
        return return_bn,return_en
    else:
        q_bn = translate_en_to_bn(query)
        results_en = search_en(query, top_k)
        results_bn = search_bn(q_bn, top_k)
        return_bn = {"queary_language": "en", "translated_query": q_bn, "results_language": "bn", "results": results_bn}
        return_en = {"queary_language": "en", "translated_query": q_bn, "results_language": "en", "results": results_en}
        return return_bn,return_en

### 4.5 Test

In [12]:
query_key= "বাংলাদেশ ক্রিকেট"
out_bn, out_en = clir_search(query_key, top_k=5)

print("Print result in Bangla:")
for r in out_bn['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")


print("Result in English:")
for r in out_en['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")

Print result in Bangla:
5.7852337630911475 বিপিএল শুরুর আগমুহূর্তে গুরুত্বপূর্ণ দায়িত্বে নান্নু - https://www.jagonews24.com/sports/news/1079007
____________________________________________
5.498008033997645 বক্সিং ডে টেস্টের নেতৃত্বে স্মিথ, নেই কামিন্স-লায়ন - https://www.risingbd.com/sports/news/633407
____________________________________________
5.3337631155419105 টিভিতে আজকের খেলা - https://www.risingbd.com/sports/news/633471
____________________________________________
2.9149818534770966 ঢাকায় পৌঁছেছে শহীদ শরিফ ওসমান হাদির মরদেহ - https://bangladeshdiplomat.com/11050/latest/%e0%a6%a2%e0%a6%be%e0%a6%95%e0%a6%be%e0%a6%af%e0%a6%bc-%e0%a6%aa%e0%a7%8c%e0%a6%81%e0%a6%9b%e0%a7%87%e0%a6%9b%e0%a7%87-%e0%a6%b6%e0%a6%b9%e0%a7%80%e0%a6%a6-%e0%a6%b6%e0%a6%b0%e0%a6%bf%e0%a6%ab/
____________________________________________
2.8633953491319146 নিরাপদ বাংলাদেশ গড়তে চাই: তারেক রহমান - https://www.risingbd.com/politics/news/633421
____________________________________________
Result in English:
5.933

In [13]:
query_key= "Bangladesh Cricket"
out_bn, out_en = clir_search(query_key, top_k=5)

print("Print result in Bangla:")
for r in out_bn['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")


print("Result in English:")
for r in out_en['results']:
    print(r['score'],r['title'],"-", r['url'])
    print("____________________________________________")

Print result in Bangla:
5.7852337630911475 বিপিএল শুরুর আগমুহূর্তে গুরুত্বপূর্ণ দায়িত্বে নান্নু - https://www.jagonews24.com/sports/news/1079007
____________________________________________
5.498008033997645 বক্সিং ডে টেস্টের নেতৃত্বে স্মিথ, নেই কামিন্স-লায়ন - https://www.risingbd.com/sports/news/633407
____________________________________________
5.3337631155419105 টিভিতে আজকের খেলা - https://www.risingbd.com/sports/news/633471
____________________________________________
2.9149818534770966 ঢাকায় পৌঁছেছে শহীদ শরিফ ওসমান হাদির মরদেহ - https://bangladeshdiplomat.com/11050/latest/%e0%a6%a2%e0%a6%be%e0%a6%95%e0%a6%be%e0%a6%af%e0%a6%bc-%e0%a6%aa%e0%a7%8c%e0%a6%81%e0%a6%9b%e0%a7%87%e0%a6%9b%e0%a7%87-%e0%a6%b6%e0%a6%b9%e0%a7%80%e0%a6%a6-%e0%a6%b6%e0%a6%b0%e0%a6%bf%e0%a6%ab/
____________________________________________
2.8633953491319146 নিরাপদ বাংলাদেশ গড়তে চাই: তারেক রহমান - https://www.risingbd.com/politics/news/633421
____________________________________________
Result in English:
5.933